# OpenRouter API Basics

This notebook demonstrates basic OpenRouter API usage: authentication, simple completions, and error handling.

**Requires:** `OPENROUTER_API_KEY` environment variable.

Uses the free model `qwen/qwen3.6-plus:free` for all examples. Free models have strict rate limits — the helper function retries automatically.

In [ ]:
import os
import time
import requests

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "Set OPENROUTER_API_KEY environment variable"

BASE_URL = "https://openrouter.ai/api/v1"
HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}


def openrouter_chat(model, messages, retries=3, delay=10):
    """Chat completion with retry on rate limit."""
    for attempt in range(retries):
        response = requests.post(
            f"{BASE_URL}/chat/completions",
            headers=HEADERS,
            json={"model": model, "messages": messages},
        )
        if response.status_code == 429:
            wait = delay * (attempt + 1)
            print(f"Rate limited, waiting {wait}s... (attempt {attempt+1}/{retries})")
            time.sleep(wait)
            continue
        response.raise_for_status()
        return response.json()
    response.raise_for_status()


print(f"API key configured (ends with ...{OPENROUTER_API_KEY[-4:]})")

## Simple Chat Completion

In [ ]:
data = openrouter_chat(
    "qwen/qwen3.6-plus:free",
    [{"role": "user", "content": "What is OpenRouter in one sentence?"}],
)
print(data["choices"][0]["message"]["content"])

## Multi-Turn Conversation

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant. Be concise."},
    {"role": "user", "content": "What are the three laws of thermodynamics?"},
]

data = openrouter_chat("qwen/qwen3.6-plus:free", messages)
assistant_msg = data["choices"][0]["message"]["content"]
print(assistant_msg)

# Follow-up
messages.append({"role": "assistant", "content": assistant_msg})
messages.append({"role": "user", "content": "Which one is most relevant to chemistry?"})

data = openrouter_chat("qwen/qwen3.6-plus:free", messages)
print("\n---\n")
print(data["choices"][0]["message"]["content"])

## Error Handling

In [ ]:
# Test with invalid model name
response = requests.post(
    f"{BASE_URL}/chat/completions",
    headers=HEADERS,
    json={
        "model": "nonexistent/model-name",
        "messages": [{"role": "user", "content": "Hello"}],
    },
)

print(f"Status: {response.status_code}")
print(f"Error: {response.json().get('error', 'none')}")

## Usage & Cost Tracking

In [ ]:
data = openrouter_chat(
    "qwen/qwen3.6-plus:free",
    [{"role": "user", "content": "Say hello in 5 languages."}],
)

print(data["choices"][0]["message"]["content"])
print("\n--- Usage ---")
usage = data.get("usage", {})
print(f"Prompt tokens:     {usage.get('prompt_tokens', 'N/A')}")
print(f"Completion tokens: {usage.get('completion_tokens', 'N/A')}")
print(f"Total tokens:      {usage.get('total_tokens', 'N/A')}")